In [ ]:
 #============================================================
# PRÉDICTION DU PRIX DES MAISONS AVEC TENSORFLOW / KERAS
# ============================================================

import pandas as pd
import numpy as np
import tensorflow as tf # type: ignore
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. CHARGEMENT DES DONNÉES
# ============================================================

df = pd.read_csv(
    "dataset_prix_maisons_300.csv"
)


# ============================================================
# 2. VÉRIFICATION
# ============================================================

print("===== DATASET =====")

print(df.head())

print("\nTaille :", df.shape)

print("\nValeurs manquantes :")

print(df.isnull().sum())


# ============================================================
# 3. X ET y
# ============================================================

X = df.drop(
    "prix_fcfa",
    axis=1
)

y = df["prix_fcfa"]


# ============================================================
# 4. TRAIN / TEST
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 5. CONVERSION EN FLOAT32
# ============================================================

X_train = X_train.astype(
    "float32"
)

X_test = X_test.astype(
    "float32"
)

y_train = y_train.astype(
    "float32"
)

y_test = y_test.astype(
    "float32"
)


# ============================================================
# 6. NORMALISATION
# ============================================================

normalizer = tf.keras.layers.Normalization()

normalizer.adapt(
    X_train
)


# ============================================================
# 7. CONSTRUCTION DU MODÈLE
# ============================================================

model = tf.keras.Sequential([

    # Normalisation
    normalizer,

    # Première couche cachée
    tf.keras.layers.Dense(
        64,
        activation="relu"
    ),

    # Deuxième couche cachée
    tf.keras.layers.Dense(
        32,
        activation="relu"
    ),

    # Couche de sortie
    tf.keras.layers.Dense(
        1
    )
])


# ============================================================
# 8. AFFICHER LE MODÈLE
# ============================================================

model.summary()


# ============================================================
# 9. COMPILATION
# ============================================================

model.compile(

    # Algorithme d'optimisation
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    # Fonction de perte
    loss="mse",

    # Métrique
    metrics=["mae"]
)


# ============================================================
# 10. CALLBACK EARLY STOPPING
# ============================================================

early_stopping = tf.keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=30,

    restore_best_weights=True
)


# ============================================================
# 11. ENTRAÎNEMENT
# ============================================================

history = model.fit(

    X_train,

    y_train,

    epochs=500,

    batch_size=32,

    validation_split=0.20,

    callbacks=[
        early_stopping
    ],

    verbose=1
)


# ============================================================
# 12. ÉVALUATION
# ============================================================

loss, mae = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\n===== ÉVALUATION =====")

print(
    f"Loss : {loss:,.2f}"
)

print(
    f"MAE : {mae:,.2f} FCFA"
)


# ============================================================
# 13. PRÉDICTIONS
# ============================================================

y_pred = model.predict(
    X_test,
    verbose=0
).flatten()


# ============================================================
# 14. MÉTRIQUES
# ============================================================

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(mse)

mae = mean_absolute_error(
    y_test,
    y_pred
)

r2 = r2_score(
    y_test,
    y_pred
)


print("\n===== MÉTRIQUES =====")

print(
    f"MAE  : {mae:,.2f} FCFA"
)

print(
    f"RMSE : {rmse:,.2f} FCFA"
)

print(
    f"R²   : {r2:.4f}"
)


# ============================================================
# 15. COURBE LOSS
# ============================================================

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    history.history["loss"],
    label="Train"
)

plt.plot(
    history.history["val_loss"],
    label="Validation"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title(
    "Évolution de la fonction de perte"
)

plt.legend()

plt.show()


# ============================================================
# 16. PRIX RÉELS VS PRÉDITS
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.scatter(
    y_test,
    y_pred
)

minimum = min(
    y_test.min(),
    y_pred.min()
)

maximum = max(
    y_test.max(),
    y_pred.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum]
)

plt.xlabel(
    "Prix réel"
)

plt.ylabel(
    "Prix prédit"
)

plt.title(
    "Prix réel vs prix prédit"
)

plt.show()


# ============================================================
# 17. NOUVELLE MAISON
# ============================================================

nouvelle_maison = pd.DataFrame({

    "surface_terrain_m2": [250],

    "surface_maison_m2": [150],

    "nombre_chambres": [4],

    "nombre_salles_bain": [2],

    "distance_centre_km": [5.0],

    "garage": [1],

    "jardin": [1],

    "piscine": [0],

    "nombre_etages": [2],

    "annee_construction": [2020]

})


# ============================================================
# 18. PRÉDICTION
# ============================================================

prix_predit = model.predict(
    nouvelle_maison,
    verbose=0
)


print("\n===== NOUVELLE MAISON =====")

print(
    f"Prix estimé : "
    f"{prix_predit[0][0]:,.0f} FCFA"
)


# ============================================================
# 19. SAUVEGARDE DU MODÈLE
# ============================================================

model.save(
    "modele_prix_maisons.keras"
)

print(
    "\nModèle sauvegardé !"
)

ModuleNotFoundError: No module named 'tensorflow'